## Install packages

In [9]:
!pip install -q pandas numpy pyarrow minio python-dotenv tqdm sentence-transformers faiss-cpu nltk groq spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 8.8 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


## Import

In [10]:
import os
import re
import json
import time
import pickle
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

from minio import Minio
from dotenv import load_dotenv

import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

import spacy
from groq import Groq

from sentence_transformers import SentenceTransformer
import faiss

## Config

In [11]:
BASE_DIR = Path("/home/jovyan/work")

RAW_DIR = BASE_DIR / "data" / "raw" / "kaggle"
CONFIG_DIR = BASE_DIR / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

JOB_POSTINGS_FILE = RAW_DIR / "linkedin_job_postings.csv"
JOB_SKILLS_FILE = RAW_DIR / "job_skills.csv"
ENV_FILE = BASE_DIR / ".env"

BRONZE_PREFIX = "bronze/kaggle"
SILVER_PREFIX = "silver/kaggle"
GOLD_PREFIX = "gold/kaggle"
INDEX_PREFIX = "index/kaggle"

WHITELIST_PATH = CONFIG_DIR / "skill_whitelist.json"
PROGRESS_PATH = CONFIG_DIR / "whitelist_progress.json"

print("Job postings exists:", JOB_POSTINGS_FILE.exists())
print("Job skills exists:", JOB_SKILLS_FILE.exists())
print("ENV exists:", ENV_FILE.exists())

Job postings exists: True
Job skills exists: True
ENV exists: True


## Connect MinIO

In [12]:
load_dotenv(ENV_FILE)

MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "minio:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY", "admin")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY", "password123")
MINIO_BUCKET = os.getenv("MINIO_BUCKET", "linkedin-data")

client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False
)

if not client.bucket_exists(MINIO_BUCKET):
    client.make_bucket(MINIO_BUCKET)
    print(f"Created bucket: {MINIO_BUCKET}")
else:
    print(f"Bucket exists: {MINIO_BUCKET}")

Bucket exists: bigdata-nhom6


## Upload raw files to MinIO bronze

In [13]:
client.fput_object(
    MINIO_BUCKET,
    f"{BRONZE_PREFIX}/linkedin_job_postings.csv",
    str(JOB_POSTINGS_FILE)
)

client.fput_object(
    MINIO_BUCKET,
    f"{BRONZE_PREFIX}/job_skills.csv",
    str(JOB_SKILLS_FILE)
)

print("Uploaded raw CSV to MinIO bronze/kaggle")

Uploaded raw CSV to MinIO bronze/kaggle


## LOAD DATA

In [14]:
jobs = pd.read_csv(JOB_POSTINGS_FILE)
skills = pd.read_csv(JOB_SKILLS_FILE)

print("jobs shape:", jobs.shape)
print("skills shape:", skills.shape)

display(jobs.head())
display(skills.head())

print("Jobs columns:", jobs.columns.tolist())
print("Skills columns:", skills.columns.tolist())

jobs shape: (1348454, 14)
skills shape: (1296381, 2)


,job_link,last_processed_time,got_summary,got_ner,is_being_worked,job_title,company,job_location,first_seen,search_city,search_country,search_position,job_level,job_type
0,https://www.linkedin.com/jobs/view/account-exe...,2024-01-21 07:12:29.00256+00,t,t,f,Account Executive - Dispensing (NorCal/Norther...,BD,"San Diego, CA",2024-01-15,Coronado,United States,Color Maker,Mid senior,Onsite
1,https://www.linkedin.com/jobs/view/registered-...,2024-01-21 07:39:58.88137+00,t,t,f,Registered Nurse - RN Care Manager,Trinity Health MI,"Norton Shores, MI",2024-01-14,Grand Haven,United States,Director Nursing Service,Mid senior,Onsite
2,https://www.linkedin.com/jobs/view/restaurant-...,2024-01-21 07:40:00.251126+00,t,t,f,RESTAURANT SUPERVISOR - THE FORKLIFT,Wasatch Adaptive Sports,"Sandy, UT",2024-01-14,Tooele,United States,Stand-In,Mid senior,Onsite
3,https://www.linkedin.com/jobs/view/independent...,2024-01-21 07:40:00.308133+00,t,t,f,Independent Real Estate Agent,Howard Hanna | Rand Realty,"Englewood Cliffs, NJ",2024-01-16,Pinehurst,United States,Real-Estate Clerk,Mid senior,Onsite
4,https://www.linkedin.com/jobs/view/group-unit-...,2024-01-19 09:45:09.215838+00,f,f,f,Group/Unit Supervisor (Systems Support Manager...,"IRS, Office of Chief Counsel","Chamblee, GA",2024-01-17,Gadsden,United States,Supervisor Travel-Information Center,Mid senior,Onsite


,job_link,job_skills
0,https://www.linkedin.com/jobs/view/housekeeper...,"Building Custodial Services, Cleaning, Janitor..."
1,https://www.linkedin.com/jobs/view/assistant-g...,"Customer service, Restaurant management, Food ..."
2,https://www.linkedin.com/jobs/view/school-base...,"Applied Behavior Analysis (ABA), Data analysis..."
3,https://www.linkedin.com/jobs/view/electrical-...,"Electrical Engineering, Project Controls, Sche..."
4,https://www.linkedin.com/jobs/view/electrical-...,"Electrical Assembly, Point to point wiring, St..."


Jobs columns: ['job_link', 'last_processed_time', 'got_summary', 'got_ner', 'is_being_worked', 'job_title', 'company', 'job_location', 'first_seen', 'search_city', 'search_country', 'search_position', 'job_level', 'job_type']
Skills columns: ['job_link', 'job_skills']


## Join Data

In [15]:
df = jobs.merge(skills, on="job_link", how="inner")

print("After join:", df.shape)
display(df.head())

After join: (1296381, 15)


,job_link,last_processed_time,got_summary,got_ner,is_being_worked,job_title,company,job_location,first_seen,search_city,search_country,search_position,job_level,job_type,job_skills
0,https://www.linkedin.com/jobs/view/account-exe...,2024-01-21 07:12:29.00256+00,t,t,f,Account Executive - Dispensing (NorCal/Norther...,BD,"San Diego, CA",2024-01-15,Coronado,United States,Color Maker,Mid senior,Onsite,"Medical equipment sales, Key competitors, Term..."
1,https://www.linkedin.com/jobs/view/registered-...,2024-01-21 07:39:58.88137+00,t,t,f,Registered Nurse - RN Care Manager,Trinity Health MI,"Norton Shores, MI",2024-01-14,Grand Haven,United States,Director Nursing Service,Mid senior,Onsite,"Nursing, Bachelor of Science in Nursing, Maste..."
2,https://www.linkedin.com/jobs/view/restaurant-...,2024-01-21 07:40:00.251126+00,t,t,f,RESTAURANT SUPERVISOR - THE FORKLIFT,Wasatch Adaptive Sports,"Sandy, UT",2024-01-14,Tooele,United States,Stand-In,Mid senior,Onsite,"Restaurant Operations Management, Inventory Ma..."
3,https://www.linkedin.com/jobs/view/independent...,2024-01-21 07:40:00.308133+00,t,t,f,Independent Real Estate Agent,Howard Hanna | Rand Realty,"Englewood Cliffs, NJ",2024-01-16,Pinehurst,United States,Real-Estate Clerk,Mid senior,Onsite,"Real Estate, Customer Service, Sales, Negotiat..."
4,https://www.linkedin.com/jobs/view/registered-...,2024-01-21 08:08:19.663033+00,t,t,f,Registered Nurse (RN),Trinity Health MI,"Muskegon, MI",2024-01-14,Muskegon,United States,Nurse Practitioner,Mid senior,Onsite,"Nursing, BSN, Medical License, Virtual RN, Nur..."


## NLP SETUP

In [16]:
for pkg in ["wordnet", "omw-1.4", "stopwords"]:
    try:
        nltk.data.find(f"corpora/{pkg}")
    except Exception:
        nltk.download(pkg, quiet=True)

lemmatizer = WordNetLemmatizer()
STOP_WORDS = set(stopwords.words("english"))

nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "tagger", "textcat"]
)

SYNONYM_MAP = {
    # Level
    "sr": "senior",
    "jr": "junior",
    "mid": "mid",
    "lead": "senior",
    "principal": "senior",
    "intern": "intern",
    "trainee": "junior",

    # Title
    "mgr": "manager",
    "mngr": "manager",
    "dir": "director",
    "vp": "vice president",
    "assoc": "associate",
    "asst": "assistant",
    "rep": "representative",
    "exec": "executive",
    "admin": "administrator",

    # Tech
    "dev": "developer",
    "eng": "engineer",
    "swe": "software engineer",
    "frontend": "front end",
    "backend": "back end",
    "fullstack": "full stack",
    "full-stack": "full stack",
    "qa": "quality assurance",
    "ml": "machine learning",
    "ai": "artificial intelligence",
    "ds": "data scientist",
    "de": "data engineer",
    "fe": "front end",
    "be": "back end",

    # Business
    "bi": "business intelligence",
    "biz": "business",
    "bd": "business development",
    "ba": "business analyst",
    "pm": "project manager",

    # Other
    "hr": "human resources",
    "cs": "customer service",
    "ae": "account executive",
    "am": "account manager",
    "mkt": "marketing",
    "seo": "search engine optimization",
    "rn": "registered nurse",
    "lpn": "licensed practical nurse",
    "np": "nurse practitioner",
    "md": "medical doctor",
    "ops": "operations",
    "coo": "chief operating officer",
    "cto": "chief technology officer",
    "ceo": "chief executive officer",
    "&": "and",
    "/": " ",
    "pt": "part time",
    "ft": "full time",

    # Skill normalization
    "powerbi": "power bi",
    "power-bi": "power bi",
    "ms excel": "excel",
    "microsoft excel": "excel",
    "sql server": "sql",
    "postgresql": "sql",
    "mysql": "sql",
    "python3": "python",
}

NOISE_PATTERNS = [
    r"\b(remote|hybrid|onsite|on.site)\b",
    r"\b(part.time|full.time|contract|temp|temporary)\b",
    r"\b(urgent|immediate|opening|opportunity|hiring|wanted)\b",
    r"\b(vietnam|viet nam|ho chi minh|hanoi|ha noi|da nang|singapore|london|new york|california|texas|florida|chicago|boston|seattle|austin|canada|australia)\b",
    r"\b(f/m/d|m/f/d|w/m/d|m/w/d)\b",
    r"\b\d+\+?\s*(year|yr|month|week)s?\b",
    r"[-|\\]+.*$",
    r"\(.*?\)",
    r"\[.*?\]",
]

NOT_JOB_TITLE = {
    "compensation", "experience", "salary", "benefit",
    "remote", "hybrid", "travel", "unknown", "other",
    "na", "n/a", "none", "null", "opening", "opportunity",
    "position", "role", "job", "hiring", "wanted"
}

VALID_SINGLE_WORD = {
    "accountant", "analyst", "architect", "consultant",
    "developer", "designer", "engineer", "manager",
    "nurse", "programmer", "recruiter", "scientist",
    "specialist", "supervisor", "teacher", "therapist",
    "coordinator", "administrator", "director", "officer",
    "technician", "mechanic", "driver", "chef", "cook",
    "attorney", "lawyer", "doctor", "pharmacist",
}

## BUILD WHITELIST

In [17]:
GROQ_KEYS = [
    k for k in [
        os.getenv("GROQ_API_KEY_1"),
        os.getenv("GROQ_API_KEY_2"),
        os.getenv("GROQ_API_KEY_3"),
        os.getenv("GROQ_API_KEY_4"),
        os.getenv("GROQ_API_KEY_5"),
        os.getenv("GROQ_API_KEY"),
    ]
    if k
]

print(f"Loaded Groq keys: {len(GROQ_KEYS)}")

SYSTEM_PROMPT = """
You are a job skill classifier.

Keep items that are genuine job skills:
- specific tools
- technologies
- programming languages
- frameworks
- certifications
- software
- equipment
- domain knowledge
- professional techniques

Remove:
- soft skills
- personality traits
- job descriptions
- physical requirements
- education requirements
- benefits
- vague phrases

When in doubt, remove it.
"""

current_key_idx = 0


def get_groq_client():
    return Groq(api_key=GROQ_KEYS[current_key_idx])


def validate_batch(batch, retry=0):
    global current_key_idx

    if not GROQ_KEYS:
        return []

    if retry >= len(GROQ_KEYS):
        return None

    try:
        response = get_groq_client().chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": (
                        "From this list, keep only genuine job skills.\n\n"
                        + ", ".join(batch)
                        + '\n\nReturn ONLY this JSON:\n{"valid_skills": ["skill1", "skill2"]}'
                    )
                }
            ],
            temperature=0.0,
            max_tokens=2048,
            response_format={"type": "json_object"}
        )

        data = json.loads(response.choices[0].message.content.strip())
        return [
            str(skill).lower().strip()
            for skill in data.get("valid_skills", [])
            if str(skill).strip()
        ]

    except Exception as e:
        err = str(e)

        if "429" in err and "tokens per day" in err:
            current_key_idx += 1
            if current_key_idx < len(GROQ_KEYS):
                print(f"Switch to Groq key {current_key_idx + 1}/{len(GROQ_KEYS)}")
                return validate_batch(batch, retry + 1)

            return None

        if "429" in err:
            print("Rate limit. Sleep 30s...")
            time.sleep(30)
            return validate_batch(batch, retry)

        print("Groq error:", e)
        return []


def save_progress(whitelist, next_idx):
    with open(PROGRESS_PATH, "w", encoding="utf-8") as f:
        json.dump(
            {
                "whitelist": sorted(list(whitelist)),
                "next_idx": next_idx
            },
            f,
            ensure_ascii=False,
            indent=2
        )


def load_progress():
    if PROGRESS_PATH.exists():
        with open(PROGRESS_PATH, "r", encoding="utf-8") as f:
            data = json.load(f)

        print(f"Resume from index: {data['next_idx']}")
        print(f"Existing whitelist: {len(data['whitelist'])}")
        return set(data["whitelist"]), data["next_idx"]

    return set(), 0


# Count skill frequency
skill_counter = Counter()

for skill_text in tqdm(df["job_skills"].dropna(), desc="Counting raw skills"):
    for skill in str(skill_text).split(","):
        skill = skill.strip().lower()
        if skill:
            skill_counter[skill] += 1

print("Unique raw skills:", len(skill_counter))

# Candidate filter before Groq
CANDIDATES = [
    skill
    for skill, count in skill_counter.most_common()
    if count >= 200
    and len(skill.split()) <= 4
    and len(skill) > 1
]

print("Candidates for Groq:", len(CANDIDATES))


if WHITELIST_PATH.exists():
    with open(WHITELIST_PATH, "r", encoding="utf-8") as f:
        SKILL_WHITELIST = set(json.load(f))

    print("Loaded existing whitelist:", len(SKILL_WHITELIST))

else:
    if not GROQ_KEYS:
        print("No Groq key found. Fallback: use top candidates as whitelist.")
        SKILL_WHITELIST = set(CANDIDATES[:5000])

    else:
        SKILL_WHITELIST, start_idx = load_progress()

        BATCH_SIZE = 200
        total_batch = len(CANDIDATES) // BATCH_SIZE + 1

        print(f"Start Groq validation: {len(CANDIDATES)} candidates")
        print(f"Total batches: {total_batch}")

        for i in range(start_idx, len(CANDIDATES), BATCH_SIZE):
            batch = CANDIDATES[i:i + BATCH_SIZE]
            batch_num = i // BATCH_SIZE + 1

            print(f"Batch {batch_num}/{total_batch}...", end=" ")

            valid = validate_batch(batch)

            if valid is None:
                save_progress(SKILL_WHITELIST, i)
                print(f"\nStopped. Progress saved at batch {batch_num}")
                break

            SKILL_WHITELIST.update(valid)
            print(f"{len(valid)}/{len(batch)} valid | total={len(SKILL_WHITELIST)}")

            if batch_num % 10 == 0:
                save_progress(SKILL_WHITELIST, i + BATCH_SIZE)

            if batch_num % 25 == 0:
                print("Sleep 60s...")
                time.sleep(60)
            else:
                time.sleep(2)

    with open(WHITELIST_PATH, "w", encoding="utf-8") as f:
        json.dump(
            sorted(list(SKILL_WHITELIST)),
            f,
            ensure_ascii=False,
            indent=2
        )

    if PROGRESS_PATH.exists():
        PROGRESS_PATH.unlink()

    print("Saved whitelist:", len(SKILL_WHITELIST))

print("Final whitelist size:", len(SKILL_WHITELIST))

Loaded Groq keys: 3


Counting raw skills:   0%|          | 0/1294296 [00:00<?, ?it/s]

Unique raw skills: 2773202
Candidates for Groq: 11313
Start Groq validation: 11313 candidates
Total batches: 57
Batch 1/57... 45/200 valid | total=45
Batch 2/57... 43/200 valid | total=88
Batch 3/57... 39/200 valid | total=126
Batch 4/57... 68/200 valid | total=192
Batch 5/57... 56/200 valid | total=248
Batch 6/57... 63/200 valid | total=311
Batch 7/57... 38/200 valid | total=349
Batch 8/57... 47/200 valid | total=396
Batch 9/57... 39/200 valid | total=435
Batch 10/57... 30/200 valid | total=465
Batch 11/57... 78/200 valid | total=542
Batch 12/57... 27/200 valid | total=569
Batch 13/57... 63/200 valid | total=631
Batch 14/57... 40/200 valid | total=670
Batch 15/57... 87/200 valid | total=757
Batch 16/57... 76/200 valid | total=833
Batch 17/57... 32/200 valid | total=865
Batch 18/57... 39/200 valid | total=904
Batch 19/57... 52/200 valid | total=956
Batch 20/57... 55/200 valid | total=1011
Batch 21/57... 53/200 valid | total=1064
Batch 22/57... 45/200 valid | total=1109
Batch 23/57... 7

## Helper functions

In [18]:
def apply_synonym(text):
    if not isinstance(text, str):
        return ""

    for k, v in SYNONYM_MAP.items():
        text = re.sub(rf"\b{re.escape(k)}\b", v, text)

    return text


def clean_tokens(text):
    tokens = [
        word
        for word in text.split()
        if word not in STOP_WORDS
    ]

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    return " ".join(tokens).strip()


def normalize_skill(skill):
    if not isinstance(skill, str):
        return ""

    skill = skill.lower().strip()
    skill = re.sub(r"[^\w\s]", " ", skill)
    skill = re.sub(r"\s+", " ", skill).strip()

    skill = apply_synonym(skill)
    skill = clean_tokens(skill)

    return skill.strip()

## Clean title

In [19]:
def remove_entities_spacy(text):
    if not isinstance(text, str):
        return ""

    doc = nlp(text.strip())

    remove_spans = [
        (ent.start_char, ent.end_char)
        for ent in doc.ents
        if ent.label_ in ["ORG", "GPE", "LOC", "NORP"]
    ]

    for start, end in sorted(remove_spans, reverse=True):
        text = text[:start] + " " + text[end:]

    return text


def process_job_title(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()

    for pattern in NOISE_PATTERNS:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)

    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    text = apply_synonym(text)
    text = clean_tokens(text)

    if not text or text in NOT_JOB_TITLE:
        return ""

    if len(text.split()) == 1 and text not in VALID_SINGLE_WORD:
        return ""

    return text


tqdm.pandas()

unique_titles = df["job_title"].dropna().unique()

print("Unique titles:", len(unique_titles))

title_entity_map = {}

for title in tqdm(unique_titles, desc="spaCy remove entities from unique titles"):
    title_entity_map[title] = remove_entities_spacy(title)

df["job_title_no_entity"] = df["job_title"].map(title_entity_map)
df["job_title_clean"] = df["job_title_no_entity"].progress_apply(process_job_title)

display(df[["job_title", "job_title_no_entity", "job_title_clean"]].head(10))

print("Empty cleaned title:", (df["job_title_clean"] == "").sum())

Unique titles: 565695


spaCy remove entities from unique titles:   0%|          | 0/565695 [00:00<?, ?it/s]

/opt/conda/lib/python3.11/site-packages/spacy/pipeline/lemmatizer.py:187: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


  0%|          | 0/1296381 [00:00<?, ?it/s]

,job_title,job_title_no_entity,job_title_clean
0,Account Executive - Dispensing (NorCal/Norther...,Account Executive - Dispensing ( ) - Becton Di...,account executive
1,Registered Nurse - RN Care Manager,Registered Nurse - RN Care Manager,registered nurse
2,RESTAURANT SUPERVISOR - THE FORKLIFT,RESTAURANT SUPERVISOR - THE,restaurant supervisor
3,Independent Real Estate Agent,Independent Real Estate Agent,independent real estate agent
4,Registered Nurse (RN),Registered Nurse (RN),registered nurse
5,Part Time- HR Generalist,Part Time- HR Generalist,
6,Store Manager,Store Manager,store manager
7,Engineering Project Coordinator,Engineering Project Coordinator,engineering project coordinator
8,Special Agent: Law/Legal Background,Special Agent:,special agent
9,"Manager, Site Operations","Manager, Site Operations",manager site operation


Empty cleaned title: 307800


## Clean location

In [20]:
def clean_location(row):
    loc = str(row.get("job_location", "")).lower().strip()
    country = str(row.get("search_country", "")).lower().strip()

    # Remote first
    if "remote" in loc:
        return "remote"

    # Vietnam
    if (
        "vietnam" in loc
        or "viet nam" in loc
        or "hanoi" in loc
        or "ha noi" in loc
        or "ho chi minh" in loc
        or "hcm" in loc
        or "da nang" in loc
        or "vietnam" in country
        or "viet nam" in country
    ):
        return "vietnam"

    # United States
    if (
        "united states" in loc
        or "usa" in loc
        or "u.s." in loc
        or loc.startswith("us ")
        or "united states" in country
        or country in ["usa", "us"]
    ):
        return "united states"

    # Use search_country if available
    if country and country not in ["nan", "none", "null", ""]:
        return country

    return "other"


df["location_clean"] = df.progress_apply(clean_location, axis=1)

display(df[["job_location", "search_country", "location_clean"]].head(10))
print(df["location_clean"].value_counts().head(20))

  0%|          | 0/1296381 [00:00<?, ?it/s]

,job_location,search_country,location_clean
0,"San Diego, CA",United States,united states
1,"Norton Shores, MI",United States,united states
2,"Sandy, UT",United States,united states
3,"Englewood Cliffs, NJ",United States,united states
4,"Muskegon, MI",United States,united states
5,"New York, NY",United States,united states
6,"London, England, United Kingdom",United Kingdom,united kingdom
7,"Winnipeg, Manitoba, Canada",Canada,canada
8,"Austin, Texas Metropolitan Area",United States,united states
9,"Knoxville, TN",United States,united states


location_clean
united states     1105407
united kingdom     108523
canada              53906
australia           28541
vietnam                 4
Name: count, dtype: int64


## Clean skills

In [21]:
def process_job_skills(text, max_words=4, use_whitelist=True):
    if not isinstance(text, str):
        return []

    text = text.replace(".", ",").replace(";", ",")
    text = re.sub(r",+", ",", text)
    text = text.lower()
    text = re.sub(r"[^\w\s,]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    phrases = list(dict.fromkeys([
        phrase.strip()
        for phrase in text.split(",")
        if phrase.strip()
    ]))

    cleaned = []

    for phrase in phrases:
        skill = normalize_skill(phrase)

        if not skill:
            continue

        if len(skill.split()) > max_words:
            continue

        if len(skill) <= 1:
            continue

        if use_whitelist and SKILL_WHITELIST and skill not in SKILL_WHITELIST:
            continue

        cleaned.append(skill)

    return sorted(list(dict.fromkeys(cleaned)))


df["skills_clean"] = df["job_skills"].progress_apply(
    lambda x: process_job_skills(x, use_whitelist=True)
)

display(df[["job_skills", "skills_clean"]].head(10))

print("Rows with empty skills:", df["skills_clean"].apply(len).eq(0).sum())

  0%|          | 0/1296381 [00:00<?, ?it/s]

,job_skills,skills_clean
0,"Medical equipment sales, Key competitors, Term...","[nursing, pharmacy]"
1,"Nursing, Bachelor of Science in Nursing, Maste...","[nursing, patient education]"
2,"Restaurant Operations Management, Inventory Ma...",[inventory management]
3,"Real Estate, Customer Service, Sales, Negotiat...",[]
4,"Nursing, BSN, Medical License, Virtual RN, Nur...","[bsn, medical license, nursing]"
5,"HR, Performance assessments, Interviews, Recru...",[]
6,"Store Management, KPI Management, Team Leaders...",[kpi management]
7,"AUTOCAD, MS Project, Building Code Compliance,...","[autocad, construction management, engineering]"
8,"FBI Special Agent, Law, Legal, Criminal Invest...",[]
9,"Management Skills, Clinical Research, Clinical...",[]


Rows with empty skills: 308011


## Save silver lên MinIO

In [22]:
# ===============================
# DATA QUALITY CHECK + FILTER
# Đặt cell này ngay trước cell "Save silver lên MinIO"
# ===============================

def is_empty_list(x):
    return not isinstance(x, list) or len(x) == 0


print("===== DATA QUALITY BEFORE FILTER =====")

total_rows = len(df)

empty_title = (
    df["job_title_clean"].isna() |
    (df["job_title_clean"].astype(str).str.strip() == "")
).sum()

empty_location = (
    df["location_clean"].isna() |
    (df["location_clean"].astype(str).str.strip() == "")
).sum()

empty_skills = df["skills_clean"].apply(is_empty_list).sum()

print(f"Total rows     : {total_rows:,}")
print(f"Empty title    : {empty_title:,}")
print(f"Empty location : {empty_location:,}")
print(f"Empty skills   : {empty_skills:,}")


# Fix location null/blank
df["location_clean"] = df["location_clean"].fillna("other")
df["location_clean"] = df["location_clean"].astype(str).str.strip().str.lower()

df.loc[
    df["location_clean"].isin(["", "nan", "none", "null"]),
    "location_clean"
] = "other"


# Filter valid rows
df_valid = df[
    (df["job_title_clean"].notna()) &
    (df["job_title_clean"].astype(str).str.strip() != "") &
    (df["skills_clean"].apply(lambda x: isinstance(x, list) and len(x) > 0))
].copy()

df_valid = df_valid.reset_index(drop=True)


print("\n===== DATA QUALITY AFTER FILTER =====")
print(f"Before : {len(df):,}")
print(f"After  : {len(df_valid):,}")
print(f"Removed: {len(df) - len(df_valid):,}")

print("\nSample valid rows:")
display(df_valid[[
    "job_title",
    "job_title_clean",
    "job_location",
    "location_clean",
    "job_skills",
    "skills_clean"
]].head(10))

===== DATA QUALITY BEFORE FILTER =====
Total rows     : 1,296,381
Empty title    : 307,800
Empty location : 0
Empty skills   : 308,011

===== DATA QUALITY AFTER FILTER =====
Before : 1,296,381
After  : 762,185
Removed: 534,196

Sample valid rows:


,job_title,job_title_clean,job_location,location_clean,job_skills,skills_clean
0,Account Executive - Dispensing (NorCal/Norther...,account executive,"San Diego, CA",united states,"Medical equipment sales, Key competitors, Term...","[nursing, pharmacy]"
1,Registered Nurse - RN Care Manager,registered nurse,"Norton Shores, MI",united states,"Nursing, Bachelor of Science in Nursing, Maste...","[nursing, patient education]"
2,RESTAURANT SUPERVISOR - THE FORKLIFT,restaurant supervisor,"Sandy, UT",united states,"Restaurant Operations Management, Inventory Ma...",[inventory management]
3,Registered Nurse (RN),registered nurse,"Muskegon, MI",united states,"Nursing, BSN, Medical License, Virtual RN, Nur...","[bsn, medical license, nursing]"
4,Store Manager,store manager,"London, England, United Kingdom",united kingdom,"Store Management, KPI Management, Team Leaders...",[kpi management]
5,Engineering Project Coordinator,engineering project coordinator,"Winnipeg, Manitoba, Canada",canada,"AUTOCAD, MS Project, Building Code Compliance,...","[autocad, construction management, engineering]"
6,Assistant Vice President Behavioral Health Sci...,assistant vice president,"Somerville, NJ",united states,"Leadership, Communication, Collaboration, Beha...",[nursing]
7,LEAD SALES ASSOCIATE-PT,senior sale associate,"Tipton, IN",united states,"Cashier, Stocker, Lead capacity, Planograms, C...","[flatbed scanner operation, planogram implemen..."
8,"Senior Associate, Tax - Product Analyst",senior associate tax,"Chicago, IL",united states,"Microsoft Office Suite, UML, Flow Charting Sof...","[microsoft office suite, sql, uml, xml]"
9,Human Resources Generalist,human resource generalist,"Merced, CA",united states,"Employee relations, Employee selection, Polici...","[adp, microsoft office, sap]"


In [23]:
silver_df = df_valid[[
    "job_link",
    "job_title",
    "job_location",
    "search_country",
    "job_skills",
    "job_title_no_entity",
    "job_title_clean",
    "location_clean",
    "skills_clean"
]].copy()

silver_df = silver_df[
    (silver_df["job_title_clean"] != "") &
    (silver_df["skills_clean"].apply(len) > 0)
].reset_index(drop=True)

print("Silver shape:", silver_df.shape)
display(silver_df.head())

silver_path = "/tmp/clean_jobs.parquet"
silver_df.to_parquet(silver_path, index=False)

client.fput_object(
    MINIO_BUCKET,
    f"{SILVER_PREFIX}/clean_jobs.parquet",
    silver_path
)

print(f"Saved silver to MinIO: {SILVER_PREFIX}/clean_jobs.parquet")

Silver shape: (762185, 9)


,job_link,job_title,job_location,search_country,job_skills,job_title_no_entity,job_title_clean,location_clean,skills_clean
0,https://www.linkedin.com/jobs/view/account-exe...,Account Executive - Dispensing (NorCal/Norther...,"San Diego, CA",United States,"Medical equipment sales, Key competitors, Term...",Account Executive - Dispensing ( ) - Becton Di...,account executive,united states,"[nursing, pharmacy]"
1,https://www.linkedin.com/jobs/view/registered-...,Registered Nurse - RN Care Manager,"Norton Shores, MI",United States,"Nursing, Bachelor of Science in Nursing, Maste...",Registered Nurse - RN Care Manager,registered nurse,united states,"[nursing, patient education]"
2,https://www.linkedin.com/jobs/view/restaurant-...,RESTAURANT SUPERVISOR - THE FORKLIFT,"Sandy, UT",United States,"Restaurant Operations Management, Inventory Ma...",RESTAURANT SUPERVISOR - THE,restaurant supervisor,united states,[inventory management]
3,https://www.linkedin.com/jobs/view/registered-...,Registered Nurse (RN),"Muskegon, MI",United States,"Nursing, BSN, Medical License, Virtual RN, Nur...",Registered Nurse (RN),registered nurse,united states,"[bsn, medical license, nursing]"
4,https://uk.linkedin.com/jobs/view/store-manage...,Store Manager,"London, England, United Kingdom",United Kingdom,"Store Management, KPI Management, Team Leaders...",Store Manager,store manager,united kingdom,[kpi management]


Saved silver to MinIO: silver/kaggle/clean_jobs.parquet


## Build gold

In [25]:
gold_df = silver_df[[
    "job_link",
    "job_title_clean",
    "location_clean",
    "skills_clean"
]].rename(columns={
    "job_title_clean": "job_title",
    "location_clean": "location",
    "skills_clean": "skills"
}).copy()

gold_df["embedding_text"] = gold_df.apply(
    lambda row: f"{row['job_title']} in {row['location']}",
    axis=1
)

gold_df = gold_df[
    (gold_df["job_title"] != "") &
    (gold_df["skills"].apply(len) > 0)
].reset_index(drop=True)

print("Gold shape:", gold_df.shape)
display(gold_df.head())

gold_path = "/tmp/model_input.parquet"
gold_df.to_parquet(gold_path, index=False)

client.fput_object(
    MINIO_BUCKET,
    f"{GOLD_PREFIX}/model_input.parquet",
    gold_path
)

print(f"Saved gold to MinIO: {GOLD_PREFIX}/model_input.parquet")

Gold shape: (762185, 5)


,job_link,job_title,location,skills,embedding_text
0,https://www.linkedin.com/jobs/view/account-exe...,account executive,united states,"[nursing, pharmacy]",account executive in united states
1,https://www.linkedin.com/jobs/view/registered-...,registered nurse,united states,"[nursing, patient education]",registered nurse in united states
2,https://www.linkedin.com/jobs/view/restaurant-...,restaurant supervisor,united states,[inventory management],restaurant supervisor in united states
3,https://www.linkedin.com/jobs/view/registered-...,registered nurse,united states,"[bsn, medical license, nursing]",registered nurse in united states
4,https://uk.linkedin.com/jobs/view/store-manage...,store manager,united kingdom,[kpi management],store manager in united kingdom


Saved gold to MinIO: gold/kaggle/model_input.parquet


## Encode title + location

In [26]:
model = SentenceTransformer("all-MiniLM-L6-v2")

texts = gold_df["embedding_text"].tolist()

embeddings = model.encode(
    texts,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.asarray(embeddings).astype("float32")

print("Embeddings shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/5955 [00:00<?, ?it/s]

Embeddings shape: (762185, 384)


## Build FAISS

In [27]:
dim = embeddings.shape[1]

index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print("FAISS index total:", index.ntotal)

FAISS index total: 762185


## Upload index + metadata lên MinIO

In [28]:
faiss_path = "/tmp/faiss.index"
metadata_path = "/tmp/metadata.pkl"

faiss.write_index(index, faiss_path)

metadata = gold_df[[
    "job_link",
    "job_title",
    "location",
    "skills",
    "embedding_text"
]].to_dict("records")

with open(metadata_path, "wb") as f:
    pickle.dump(metadata, f)

client.fput_object(
    MINIO_BUCKET,
    f"{INDEX_PREFIX}/faiss.index",
    faiss_path
)

client.fput_object(
    MINIO_BUCKET,
    f"{INDEX_PREFIX}/metadata.pkl",
    metadata_path
)

print(f"Saved FAISS index to MinIO: {INDEX_PREFIX}/faiss.index")
print(f"Saved metadata to MinIO: {INDEX_PREFIX}/metadata.pkl")

Saved FAISS index to MinIO: index/kaggle/faiss.index
Saved metadata to MinIO: index/kaggle/metadata.pkl


## Luu local

In [11]:
MODEL_DIR = BASE_DIR / "models" / "kaggle"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

client.fget_object(
    MINIO_BUCKET,
    "index/kaggle/faiss.index",
    str(MODEL_DIR / "faiss.index")
)

client.fget_object(
    MINIO_BUCKET,
    "index/kaggle/metadata.pkl",
    str(MODEL_DIR / "metadata.pkl")
)

print("Downloaded index to local")

Downloaded index to local
